# Leakage-Safe Energy Forecasting with Prophet

This notebook is built for Kaggle and for mentor review. It keeps two issues separate:

1. Data leakage during training and validation: features must not use the current target or future target values.
2. Forecasting reality at inference: future lag values are not available for multi-step forecasting.

Prophet note: Kaggle GPU can be enabled, but Prophet itself is CPU-oriented. The notebook runs fine in a GPU session, but the GPU will not materially accelerate Prophet training.

In [ ]:
import os
import glob
import math
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

try:
    from prophet import Prophet
except Exception as exc:
    print("Prophet import failed. Attempting pip install. If Kaggle Internet is off, enable it or use a Kaggle image with prophet installed.")
    print(repr(exc))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "prophet"])
    from prophet import Prophet

try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    print(gpu.stdout[:1000] if gpu.returncode == 0 else "No GPU reported by nvidia-smi.")
except Exception:
    print("nvidia-smi not available. This is okay: Prophet does not need GPU.")

## 1. Load the data

On Kaggle, upload `cleaned_energy_data_model.csv` as a dataset. The path is auto-detected under `/kaggle/input`. Locally, the notebook also checks the Windows path used while creating this file.

In [ ]:
DATA_PATH = os.environ.get("DATA_PATH", "")

candidate_paths = [
    DATA_PATH,
    "/kaggle/input/cleaned-energy-data-model/cleaned_energy_data_model.csv",
    "/kaggle/input/energy-consumption/cleaned_energy_data_model.csv",
    r"C:\Users\Chavda\Downloads\cleaned_energy_data_model.csv",
]

if not DATA_PATH:
    candidate_paths.extend(glob.glob("/kaggle/input/**/cleaned_energy_data_model.csv", recursive=True))
    candidate_paths.extend(glob.glob("/kaggle/input/**/*.csv", recursive=True))

DATA_PATH = next((p for p in candidate_paths if p and os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find cleaned_energy_data_model.csv. Set DATA_PATH manually.")

raw = pd.read_csv(DATA_PATH)
print(DATA_PATH)
print(raw.shape)
display(raw.head())
display(raw.dtypes)

In [ ]:
TIME_COL = "start_time"
TARGET_COL = "consumption"
FREQ = "h"

df = raw.copy()
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
df = df.dropna(subset=[TIME_COL, TARGET_COL]).sort_values(TIME_COL)
df = df.drop_duplicates(subset=[TIME_COL], keep="last")
df = df.rename(columns={TIME_COL: "ds", TARGET_COL: "y"})[["ds", "y"]]

# Backward-looking missing handling only. If a missing hour appears, forward-fill uses only past observed y.
full_index = pd.date_range(df["ds"].min(), df["ds"].max(), freq=FREQ)
df = df.set_index("ds").reindex(full_index).rename_axis("ds").reset_index()
df["y"] = df["y"].ffill()
df = df.dropna(subset=["y"]).reset_index(drop=True)

print(df.shape)
print(df["ds"].min(), "to", df["ds"].max())
display(df.head())
display(df.tail())

## 2. Leakage audit of the provided columns

The source CSV already contains lag and rolling columns. For a production-safe notebook, we do not trust precomputed target-derived features unless their provenance is guaranteed.

Dropped for Prophet:

- Lag features: safe only for one-step prediction or horizon-specific direct models. They are not known for arbitrary future Prophet rows.
- Rolling/EMA features: safe only if computed as `y.shift(1).rolling(...)` or `y.shift(1).ewm(...)`; otherwise they include current `y`.
- Target transforms such as log/sqrt/smooth: never use as input regressors for the same target row.
- Deviation features using `y - group_mean`: not future-known for Prophet because they require future `y`.

Kept/rebuilt:

- Timestamp features: hour, day of week, month, weekend, working hour, peak flags, cyclic encodings.
- Train-only calendar expected-value regressors: historical averages by hour/day/month computed only in `fit()`.

In [ ]:
all_source_columns = set(raw.columns)

lag_like = sorted([c for c in all_source_columns if c.startswith("lag_")])
rolling_like = sorted([c for c in all_source_columns if c.startswith("rolling_") or c.startswith("ewm_") or c.startswith("ema_")])
target_transforms = sorted([c for c in all_source_columns if c.startswith("consumption_")])
deviation_like = sorted([c for c in all_source_columns if "deviation" in c.lower() or "resid" in c.lower()])

audit = pd.DataFrame(
    [
        ["lag_like", lag_like, "Do not use in Prophet future dataframe; use only in horizon-aware direct models."],
        ["rolling_like", rolling_like, "Recompute from y.shift(1) if using ML lags; not used as Prophet regressors."],
        ["target_transforms", target_transforms, "Use only as alternative target, never as same-row input."],
        ["deviation_like", deviation_like, "Unsafe if computed from full data or requires current/future y."],
    ],
    columns=["group", "columns_found", "decision"],
)
display(audit)

## 3. Leakage-safe preprocessor

This class follows the training/inference rule:

- `fit(train_df)` can look at `y` only inside the training window.
- `transform(any_df)` creates features from timestamps plus stored training statistics.
- Future rows do not need future `y`.

## 3A. Outlier handling without leakage

Outliers are handled only inside the training window. We do not cap or delete validation/test targets before scoring, because that would make the metric dishonest.

For electricity data, many spikes are real demand events, not errors. The safest rule is:

- Keep plausible seasonal peaks and troughs.
- Remove or ignore impossible sensor errors, such as negative values, zeros if impossible, or extreme isolated jumps.
- Fit outlier thresholds on training data only.
- For Prophet, set training outlier `y` values to `NaN`; Prophet will ignore those points during fitting while still predicting those timestamps.


In [ ]:
class RobustOutlierHandler:
    def __init__(self, group_cols=("month", "hour"), z_limit=5.0, min_group_size=30):
        self.group_cols = list(group_cols)
        self.z_limit = z_limit
        self.min_group_size = min_group_size
        self.global_median_ = None
        self.global_mad_ = None
        self.group_stats_ = None

    @staticmethod
    def _calendar(frame):
        out = frame[["ds", "y"]].copy()
        out["hour"] = out["ds"].dt.hour
        out["day_of_week"] = out["ds"].dt.dayofweek
        out["month"] = out["ds"].dt.month
        return out

    @staticmethod
    def _mad(values):
        median = np.nanmedian(values)
        mad = np.nanmedian(np.abs(values - median))
        return median, max(float(mad), 1e-9)

    def fit(self, train_df):
        tmp = self._calendar(train_df)
        self.global_median_, self.global_mad_ = self._mad(tmp["y"].values)

        rows = []
        for keys, grp in tmp.groupby(self.group_cols):
            if len(grp) < self.min_group_size:
                continue
            median, mad = self._mad(grp["y"].values)
            if not isinstance(keys, tuple):
                keys = (keys,)
            rows.append(dict(zip(self.group_cols, keys), median=median, mad=mad))
        self.group_stats_ = pd.DataFrame(rows)
        return self

    def flag(self, frame):
        if self.global_median_ is None:
            raise RuntimeError("Call fit(train_df) before flag().")

        tmp = self._calendar(frame)
        tmp = tmp.merge(self.group_stats_, on=self.group_cols, how="left")
        tmp["median"] = tmp["median"].fillna(self.global_median_)
        tmp["mad"] = tmp["mad"].fillna(self.global_mad_).clip(lower=1e-9)
        tmp["robust_z"] = 0.6745 * (tmp["y"] - tmp["median"]) / tmp["mad"]

        tmp["is_impossible"] = tmp["y"] <= 0
        tmp["is_statistical_outlier"] = tmp["robust_z"].abs() > self.z_limit
        tmp["is_outlier"] = tmp["is_impossible"] | tmp["is_statistical_outlier"]
        return tmp[["ds", "robust_z", "is_impossible", "is_statistical_outlier", "is_outlier"]]

    def clean_training_target(self, train_df):
        flags = self.flag(train_df)
        cleaned = train_df.copy().merge(flags[["ds", "robust_z", "is_outlier"]], on="ds", how="left")
        cleaned["y_original"] = cleaned["y"]
        cleaned.loc[cleaned["is_outlier"].fillna(False), "y"] = np.nan
        return cleaned


In [ ]:
class TimeSeriesPreprocessor:
    def __init__(self):
        self.global_mean_ = None
        self.hour_mean_ = None
        self.dow_mean_ = None
        self.month_mean_ = None
        self.hour_dow_mean_ = None
        self.regressor_cols_ = None

    @staticmethod
    def _calendar(ds):
        out = pd.DataFrame({"ds": pd.to_datetime(ds)})
        out["hour"] = out["ds"].dt.hour
        out["day_of_week"] = out["ds"].dt.dayofweek
        out["month"] = out["ds"].dt.month
        out["day_of_year"] = out["ds"].dt.dayofyear
        out["is_weekend"] = out["day_of_week"].isin([5, 6]).astype(int)
        out["is_working_hour"] = out["hour"].between(8, 18).astype(int)
        out["is_peak_morning"] = out["hour"].between(7, 10).astype(int)
        out["is_peak_evening"] = out["hour"].between(17, 21).astype(int)
        out["is_month_start"] = out["ds"].dt.is_month_start.astype(int)
        out["is_month_end"] = out["ds"].dt.is_month_end.astype(int)
        out["season"] = ((out["month"] % 12) // 3).astype(int)
        out["weekend_hour"] = out["is_weekend"] * out["hour"]
        out["season_hour"] = out["season"] * out["hour"]

        out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
        out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
        out["dow_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7)
        out["dow_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7)
        out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
        out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
        out["doy_sin"] = np.sin(2 * np.pi * out["day_of_year"] / 366)
        out["doy_cos"] = np.cos(2 * np.pi * out["day_of_year"] / 366)
        return out

    def fit(self, train_df):
        train = train_df[["ds", "y"]].copy()
        cal = self._calendar(train["ds"])
        joined = cal.join(train["y"])

        self.global_mean_ = float(joined["y"].mean())
        self.hour_mean_ = joined.groupby("hour")["y"].mean()
        self.dow_mean_ = joined.groupby("day_of_week")["y"].mean()
        self.month_mean_ = joined.groupby("month")["y"].mean()
        self.hour_dow_mean_ = joined.groupby(["hour", "day_of_week"])["y"].mean()

        transformed = self.transform(train)
        self.regressor_cols_ = [c for c in transformed.columns if c not in ["ds"]]
        return self

    def transform(self, frame):
        if self.global_mean_ is None:
            raise RuntimeError("Call fit(train_df) before transform().")

        cal = self._calendar(frame["ds"])
        cal["expected_by_hour"] = cal["hour"].map(self.hour_mean_).fillna(self.global_mean_)
        cal["expected_by_dow"] = cal["day_of_week"].map(self.dow_mean_).fillna(self.global_mean_)
        cal["expected_by_month"] = cal["month"].map(self.month_mean_).fillna(self.global_mean_)

        idx = pd.MultiIndex.from_frame(cal[["hour", "day_of_week"]])
        cal["expected_by_hour_dow"] = pd.Series(idx.map(self.hour_dow_mean_), index=cal.index).fillna(self.global_mean_)
        return cal


def add_regressors(model, regressor_cols):
    for col in regressor_cols:
        model.add_regressor(col, standardize="auto")
    return model


def make_prophet_model(regressor_cols, interval_width=0.90):
    model = Prophet(
        growth="linear",
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=True,
        seasonality_mode="multiplicative",
        interval_width=interval_width,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10.0,
    )
    model.add_seasonality(name="monthly", period=30.5, fourier_order=5)
    return add_regressors(model, regressor_cols)

## 4. Chronological train/validation/test split

No random split. The validation block is 20 percent of the six-year dataset, which gives roughly 1.2 years for seasonal validation. The final 15 percent is kept as test.

In [ ]:
train_end = int(len(df) * 0.65)
val_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_df), len(val_df), len(test_df)],
        "start": [train_df["ds"].min(), val_df["ds"].min(), test_df["ds"].min()],
        "end": [train_df["ds"].max(), val_df["ds"].max(), test_df["ds"].max()],
    }
)
display(split_summary)

In [ ]:
outlier_handler = RobustOutlierHandler(group_cols=("month", "hour"), z_limit=5.0).fit(train_df)
train_clean_df = outlier_handler.clean_training_target(train_df)
val_outlier_flags_for_analysis = outlier_handler.flag(val_df)
test_outlier_flags_for_analysis = outlier_handler.flag(test_df)

print(f"Training points ignored by Prophet as outliers: {int(train_clean_df['is_outlier'].sum())} / {len(train_clean_df)}")
display(train_clean_df.loc[train_clean_df["is_outlier"].fillna(False), ["ds", "y_original", "robust_z"]].head(10))

plt.figure(figsize=(14, 4))
plt.plot(train_df["ds"], train_df["y"], linewidth=0.5, label="training y")
plt.scatter(
    train_clean_df.loc[train_clean_df["is_outlier"].fillna(False), "ds"],
    train_clean_df.loc[train_clean_df["is_outlier"].fillna(False), "y_original"],
    s=10,
    color="red",
    label="ignored outliers",
)
plt.title("Training-only robust outlier detection")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return {
        "MAE": np.mean(np.abs(y_true - y_pred)),
        "RMSE": np.sqrt(np.mean((y_true - y_pred) ** 2)),
        "MAPE": np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100,
        "sMAPE": np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred))) * 100,
    }


pre = TimeSeriesPreprocessor().fit(train_clean_df)
regressors = pre.regressor_cols_
print(f"{len(regressors)} Prophet regressors")
print(regressors)

train_model_df = pre.transform(train_clean_df)
train_model_df["y"] = train_clean_df["y"].values

val_features = pre.transform(val_df)

model = make_prophet_model(regressors)
model.fit(train_model_df[["ds", "y"] + regressors])

val_forecast = model.predict(val_features[["ds"] + regressors])
val_eval = val_df[["ds", "y"]].merge(val_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]], on="ds")
display(pd.DataFrame([metrics(val_eval["y"], val_eval["yhat"])]))
display(val_eval.head())

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(train_df["ds"].tail(24 * 30), train_df["y"].tail(24 * 30), label="train tail", linewidth=1)
plt.plot(val_eval["ds"], val_eval["y"], label="validation actual", linewidth=1)
plt.plot(val_eval["ds"], val_eval["yhat"], label="validation forecast", linewidth=1)
plt.fill_between(val_eval["ds"], val_eval["yhat_lower"], val_eval["yhat_upper"], alpha=0.2, label="Prophet interval")
plt.title("Chronological validation forecast")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Test evaluation with refit on train + validation

The preprocessor is refit on `train + validation` only, then applied to test. This mirrors real forecasting: all statistics are learned before the test boundary.

In [ ]:
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
outlier_handler_tv = RobustOutlierHandler(group_cols=("month", "hour"), z_limit=5.0).fit(train_val_df)
train_val_clean_df = outlier_handler_tv.clean_training_target(train_val_df)

pre_tv = TimeSeriesPreprocessor().fit(train_val_clean_df)
regressors_tv = pre_tv.regressor_cols_

train_val_model_df = pre_tv.transform(train_val_clean_df)
train_val_model_df["y"] = train_val_clean_df["y"].values

test_features = pre_tv.transform(test_df)

test_model = make_prophet_model(regressors_tv)
test_model.fit(train_val_model_df[["ds", "y"] + regressors_tv])

test_forecast = test_model.predict(test_features[["ds"] + regressors_tv])
test_eval = test_df[["ds", "y"]].merge(test_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]], on="ds")
display(pd.DataFrame([metrics(test_eval["y"], test_eval["yhat"])]))
display(test_eval.head())

## 6. Walk-forward cross-validation

This uses expanding windows. It is slower than one holdout split because Prophet is refit for each fold, but it gives a more honest performance distribution.

In [ ]:
RUN_WALK_FORWARD = True
MAX_FOLDS = 3
INITIAL_DAYS = 730
HORIZON_HOURS = 24 * 30
PERIOD_DAYS = 120


def walk_forward_cv(data, initial_days=730, horizon_hours=720, period_days=120, max_folds=3):
    data = data.sort_values("ds").reset_index(drop=True)
    first_ds = data["ds"].min()
    last_possible_cutoff = data["ds"].max() - pd.Timedelta(hours=horizon_hours)

    cutoffs = []
    cutoff = first_ds + pd.Timedelta(days=initial_days)
    while cutoff <= last_possible_cutoff:
        cutoffs.append(cutoff)
        cutoff = cutoff + pd.Timedelta(days=period_days)

    if max_folds:
        cutoffs = cutoffs[-max_folds:]

    fold_metrics = []
    fold_predictions = []

    for fold, cutoff in enumerate(cutoffs, start=1):
        fold_train = data[data["ds"] <= cutoff].copy()
        fold_valid = data[(data["ds"] > cutoff) & (data["ds"] <= cutoff + pd.Timedelta(hours=horizon_hours))].copy()
        if len(fold_train) < 24 * 365 or len(fold_valid) == 0:
            continue

        fold_outliers = RobustOutlierHandler(group_cols=("month", "hour"), z_limit=5.0).fit(fold_train)
        fold_train_clean = fold_outliers.clean_training_target(fold_train)

        fold_pre = TimeSeriesPreprocessor().fit(fold_train_clean)
        fold_regressors = fold_pre.regressor_cols_

        fold_train_model = fold_pre.transform(fold_train_clean)
        fold_train_model["y"] = fold_train_clean["y"].values
        fold_valid_features = fold_pre.transform(fold_valid)

        fold_model = make_prophet_model(fold_regressors)
        fold_model.fit(fold_train_model[["ds", "y"] + fold_regressors])

        pred = fold_model.predict(fold_valid_features[["ds"] + fold_regressors])
        joined = fold_valid[["ds", "y"]].merge(pred[["ds", "yhat", "yhat_lower", "yhat_upper"]], on="ds")
        joined["fold"] = fold
        joined["cutoff"] = cutoff
        joined["horizon"] = ((joined["ds"] - cutoff) / pd.Timedelta(hours=1)).astype(int)

        row = {"fold": fold, "cutoff": cutoff, "train_rows": len(fold_train), "valid_rows": len(fold_valid)}
        row.update(metrics(joined["y"], joined["yhat"]))
        fold_metrics.append(row)
        fold_predictions.append(joined)
        print(f"Fold {fold}: cutoff={cutoff}, MAPE={row['MAPE']:.3f}%")

    metrics_df = pd.DataFrame(fold_metrics)
    preds_df = pd.concat(fold_predictions, ignore_index=True) if fold_predictions else pd.DataFrame()
    return metrics_df, preds_df


if RUN_WALK_FORWARD:
    cv_metrics, cv_predictions = walk_forward_cv(
        df,
        initial_days=INITIAL_DAYS,
        horizon_hours=HORIZON_HOURS,
        period_days=PERIOD_DAYS,
        max_folds=MAX_FOLDS,
    )
    display(cv_metrics)
    if len(cv_metrics):
        display(cv_metrics[["MAE", "RMSE", "MAPE", "sMAPE"]].agg(["mean", "std"]))
else:
    cv_metrics, cv_predictions = pd.DataFrame(), pd.DataFrame()

## 7. Conformal prediction intervals

For mentor review, use fold residuals to calibrate intervals. The interval width is horizon-specific when walk-forward predictions are available. This reflects the fact that uncertainty should increase as the forecast horizon grows.

In [ ]:
CONFORMAL_ALPHA = 0.10
MAX_CONFORMAL_HORIZON = 24


def conformal_scores(predictions, alpha=0.10, max_horizon=24):
    if predictions is None or len(predictions) == 0:
        return pd.DataFrame()

    tmp = predictions.copy()
    tmp["abs_error"] = (tmp["y"] - tmp["yhat"]).abs()
    tmp = tmp[(tmp["horizon"] >= 1) & (tmp["horizon"] <= max_horizon)]
    q = 1 - alpha / 2
    out = tmp.groupby("horizon")["abs_error"].quantile(q).rename("q_abs_error").reset_index()

    global_q = tmp["abs_error"].quantile(q)
    all_h = pd.DataFrame({"horizon": range(1, max_horizon + 1)})
    out = all_h.merge(out, on="horizon", how="left")
    out["q_abs_error"] = out["q_abs_error"].fillna(global_q)
    return out


scores = conformal_scores(cv_predictions, alpha=CONFORMAL_ALPHA, max_horizon=MAX_CONFORMAL_HORIZON)
display(scores)

## 8. Final future forecast from the main CSV only

This refits Prophet on all observed history from `cleaned_energy_data_model.csv` and creates a next-24-hour forecast after the final timestamp. No separate `test.csv` or `sample_submission.csv` is required.

In [ ]:
FINAL_FORECAST_HOURS = 24

outlier_handler_full = RobustOutlierHandler(group_cols=("month", "hour"), z_limit=5.0).fit(df)
full_clean_df = outlier_handler_full.clean_training_target(df)

pre_full = TimeSeriesPreprocessor().fit(full_clean_df)
regressors_full = pre_full.regressor_cols_

full_model_df = pre_full.transform(full_clean_df)
full_model_df["y"] = full_clean_df["y"].values

final_model = make_prophet_model(regressors_full)
final_model.fit(full_model_df[["ds", "y"] + regressors_full])

future = pd.DataFrame(
    {"ds": pd.date_range(df["ds"].max() + pd.Timedelta(hours=1), periods=FINAL_FORECAST_HOURS, freq=FREQ)}
)
future_features = pre_full.transform(future)
future_forecast = final_model.predict(future_features[["ds"] + regressors_full])
future_out = future_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()

if len(scores):
    future_out["horizon"] = np.arange(1, len(future_out) + 1)
    future_out = future_out.merge(scores, on="horizon", how="left")
    future_out["conformal_lower"] = future_out["yhat"] - future_out["q_abs_error"]
    future_out["conformal_upper"] = future_out["yhat"] + future_out["q_abs_error"]

future_out.to_csv("prophet_next_24h_forecast.csv", index=False)
display(future_out)
print("Wrote prophet_next_24h_forecast.csv")

In [ ]:
print("This notebook uses only cleaned_energy_data_model.csv.")
print("No test.csv is required. The final output is prophet_next_24h_forecast.csv.")

## 9. If you switch back to lag-based ML

Prophet avoids the future-lag problem by not requiring future lag regressors. If you later use XGBoost, LightGBM, or RandomForest with lag features, use a direct multi-step design:

- Model 1 can use `lag_1h`, `lag_24h`, `lag_168h`, etc.
- Model 24 must drop `lag_1h` through `lag_23h`, because those require unknown future targets.
- Rolling features must be computed from the origin time using `y.shift(1)` before rolling.

In [ ]:
def available_lag_columns_for_horizon(horizon, lag_sizes=(1, 2, 3, 6, 12, 24, 48, 72, 168, 336)):
    return [f"lag_{lag}h" for lag in lag_sizes if lag >= horizon]


for h in [1, 2, 6, 12, 24, 48]:
    print(f"h={h:>2}: {available_lag_columns_for_horizon(h)}")

## Mentor talking points

How leakage was handled:

- No random split; all validation/test splits are chronological.
- Precomputed target-derived features from the CSV are audited but not used as Prophet regressors.
- Rolling and EMA features would need `y.shift(1)` before rolling; this notebook avoids them for Prophet.
- Calendar expected-value regressors are fit only on the training window and then applied to validation/test/future timestamps.
- Target transforms are not used as input features.

How inference reality was handled:

- Prophet forecasts all future timestamps directly from trend, seasonality, and known future regressors.
- The future dataframe contains no future `y`, no lag values that require future `y`, and no deviations that require future `y`.

How validation was handled:

- One chronological validation block is shown for quick inspection.
- Expanding-window CV estimates fold-to-fold performance variance.
- Conformal intervals are calibrated from walk-forward residuals.